In [14]:
import fitz  # PyMuPDF
import numpy as np
from pathlib import Path
import re
from anthropic import RateLimitError
from openai import OpenAI
from anthropic import Anthropic
from collections import Counter
from sentence_transformers import SentenceTransformer

import time 

anthropic_client = Anthropic()
openai_client = OpenAI()

import logging
from pathlib import Path

import spacy

# ─────────────────────────────────────────────────────────────────────────────
# Logging — configured once at module level
# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
    handlers=[
        logging.FileHandler("pipeline.log"),
        logging.StreamHandler()
    ]
)


# ─────────────────────────────────────────────────────────────────────────────
# spaCy — loaded once at module level, reused across all files
# ─────────────────────────────────────────────────────────────────────────────
nlp = spacy.load("en_core_web_sm")
nlp.disable_pipes(["ner", "tagger"])   # only the parser is needed for sentence splitting

# ─────────────────────────────────────────────────────────────────────────────
# Sentence embedding model — loaded once, reused across all semantic refinement calls
# ─────────────────────────────────────────────────────────────────────────────
_embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Reference section headings — checked case-insensitively during extraction
REFERENCE_HEADINGS = {"references", "bibliography", "works cited"}

# A block's font size must exceed body size by this factor to be a heading candidate
HEADING_SIZE_THRESHOLD = 1.15

# Blocks with fewer than this many sentences are merged into their neighbour
MIN_BLOCK_SENTENCES = 3


2026-03-03 21:06:54,781 — INFO — Use pytorch device_name: cpu
2026-03-03 21:06:54,782 — INFO — Load pretrained SentenceTransformer: all-MiniLM-L6-v2


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# OpenAI Embeddings with Batching + Token Guard  [UPDATED: retry logic + logging]
# ─────────────────────────────────────────────────────────────────────────────

def get_embeddings(
    texts: list[str],
    model: str = "text-embedding-3-small",
    batch_size: int = 500,
) -> list[list[float]]:
    """
    Gets embeddings from OpenAI in safe batches.

    Fixes applied (original):
      - Batch size of 500 stays well under the 2048 input limit per request
      - Truncates any single text exceeding ~8000 tokens (1 token ≈ 4 chars)
        to prevent BadRequestError on long academic sentences / footnotes
      - Tracks truncated texts so you know which sentences were cut

    Fixes applied (this update):
      - Retry logic: exponential backoff on RateLimitError (up to 3 attempts),
        consistent with the same pattern used in add_contextual_retrieval.
        A failed batch after 3 attempts raises so the caller knows explicitly —
        silent partial embeddings are worse than a loud failure.
      - All print() calls replaced with logging.* so embedding progress appears
        in pipeline.log alongside all other pipeline output.

    Known future improvements (not applied yet):
      - Truncation at sentence boundary instead of raw character index
      - Non-ASCII-aware token estimation for heavy LaTeX/CJK content
    """
    MAX_CHARS = 32000  # ~8000 tokens — OpenAI's per-input hard limit

    cleaned = []
    truncated_count = 0
    for t in texts:
        t = t.replace("\n", " ")
        if len(t) > MAX_CHARS:
            t = t[:MAX_CHARS]
            truncated_count += 1
        cleaned.append(t)

    if truncated_count:
        logging.warning(f"get_embeddings — {truncated_count} text(s) truncated to fit OpenAI token limit.")

    all_embeddings = []
    total_batches = -(-len(cleaned) // batch_size)   # ceiling division

    for i in range(0, len(cleaned), batch_size):
        batch     = cleaned[i: i + batch_size]
        batch_num = i // batch_size + 1

        logging.info(f"get_embeddings — batch {batch_num}/{total_batches} ({len(batch)} texts)…")

        # ── Retry with exponential backoff on rate limit ──────────────────────
        response = None
        for attempt in range(3):
            try:
                response = openai_client.embeddings.create(input=batch, model=model)
                break
            except RateLimitError:
                wait = 2 ** attempt   # 1s → 2s → 4s
                logging.warning(
                    f"get_embeddings — rate limited on batch {batch_num} "
                    f"(attempt {attempt + 1}/3), retrying in {wait}s…"
                )
                time.sleep(wait)

        if response is None:
            # All 3 attempts exhausted — raise explicitly so the caller can decide
            # whether to skip this document or abort the pipeline entirely.
            raise RuntimeError(
                f"get_embeddings — batch {batch_num}/{total_batches} failed after 3 attempts. "
                "Check your OpenAI rate limits or API key."
            )

        all_embeddings.extend([item.embedding for item in response.data])

    return all_embeddings

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — PDF Text Extraction
# ─────────────────────────────────────────────────────────────────────────────
def extract_text_from_pdf(pdf_path: str) -> list[dict] | None:
    """
    Extracts and cleans text from each page of a PDF.

    Layout cleaning applied at extraction time (PyMuPDF's responsibility):
      - Strips headers and footers using bounding box Y position
      - Drops table/figure noise (short numeric-only blocks)
      - Rejoins hyphenated line breaks from two-column layouts
      - Joins stray mid-sentence newlines where next line starts lowercase
      - Detects and discards the references/bibliography section entirely

    Returns None if no extractable text is found (fully scanned/image PDF).
    """
    doc = fitz.open(pdf_path)
    pages = []
    empty_pages = []
    references_reached = False   # flag — once set, all subsequent blocks are dropped

    for i, page in enumerate(doc):
        if references_reached:
            break                # no point reading further pages once references start

        page_height = page.rect.height
        blocks = page.get_text("blocks")   # each block: (x0, y0, x1, y1, text, ...)
        lines = []

        for block in blocks:
            block_text = block[4].strip()
            block_y_top = block[1]
            block_y_bottom = block[3]

            # ── References section detection ──────────────────────────────────
            # Check the first line of each block against known reference headings.
            # Once found, set the flag and stop processing entirely.
            first_line = block_text.split("\n")[0].strip().lower()
            if first_line in REFERENCE_HEADINGS:
                logging.info(f"'{pdf_path}' — references section detected at page {i + 1}, discarding tail.")
                references_reached = True
                break

            # ── Strip headers and footers ─────────────────────────────────────
            # Blocks in top 7% or bottom 7% of the page are headers/footers.
            if block_y_top < page_height * 0.07:
                logging.debug(f"Stripped header block: '{block_text[:50]}'")
                continue
            if block_y_bottom > page_height * 0.93:
                logging.debug(f"Stripped footer block: '{block_text[:50]}'")
                continue

            # ── Drop table and figure noise ───────────────────────────────────
            # Short blocks that are purely numeric/punctuation with no real words.
            if re.fullmatch(r'[\d\s\.\,\%\-]+', block_text) and len(block_text) < 40:
                logging.debug(f"Stripped table noise: '{block_text[:50]}'")
                continue

            # Split block into individual lines for hyphenation and newline fixing
            lines.extend(block_text.split("\n"))

        if references_reached:
            break

        # ── Rejoin hyphenated line breaks ─────────────────────────────────────
        # "large-\nscale" → "large-scale"
        # Strip the trailing hyphen and attach the next line directly (no space).
        rejoined = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            if rejoined and rejoined[-1].endswith("-"):
                rejoined[-1] = rejoined[-1][:-1] + line
            else:
                rejoined.append(line)

        # ── Join stray mid-sentence newlines ──────────────────────────────────
        # If a line doesn't end with sentence-closing punctuation and the next
        # line starts with a lowercase letter, it's a mid-sentence wrap — join
        # with a space. If the next line starts with a capital, leave separate.
        cleaned_lines = []
        for j, line in enumerate(rejoined):
            if (
                cleaned_lines
                and not cleaned_lines[-1][-1] in ".!?"
                and line
                and line[0].islower()
            ):
                cleaned_lines[-1] = cleaned_lines[-1] + " " + line
            else:
                cleaned_lines.append(line)

        text = " ".join(cleaned_lines).strip()

        if text:
            pages.append({"page_num": i + 1, "text": text})
        else:
            empty_pages.append(i + 1)

    doc.close()

    if empty_pages:
        logging.warning(f"'{pdf_path}' — pages with no extractable text: {empty_pages}")

    if not pages:
        logging.error(
            f"'{pdf_path}' — fully scanned/image-based PDF, no text extracted. "
            "Consider OCR (e.g. pytesseract or AWS Textract)."
        )
        return None

    return pages

In [4]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Sentence Splitting
# ─────────────────────────────────────────────────────────────────────────────

def split_into_sentences(text: str, max_sentence_chars: int = 2048) -> list[str]:
    """
    Splits cleaned text into sentences using spaCy's dependency parser.

    spaCy handles linguistic ambiguity that regex and NLTK cannot:
      - Abbreviations: Dr., et al., Fig., U.S.A.
      - Inline citations: (Smith et al., 2019)
      - Decimal numbers: 96.5, λ = 0.01
      - Complex punctuation in academic text

    Fallback: any sentence exceeding max_sentence_chars is split at the
    nearest whitespace to its midpoint — handles no-punctuation blocks
    that spaCy returns as one oversized unit.

    Layout problems (hyphenation, headers, footers, cross-page boundaries)
    are already cleaned upstream before this function is called.
    """
    doc = nlp(text)
    sentences = []

    for sent in doc.sents:
        s = sent.text.strip()

        if not s or not re.search(r'[a-zA-Z0-9]', s):
            continue   # filter empty or symbol-only fragments

        # ── Oversized sentence fallback ───────────────────────────────────────
        # If spaCy returns a giant block with no punctuation, split it at the
        # nearest whitespace to the midpoint rather than keeping one huge unit.
        if len(s) > max_sentence_chars:
            midpoint = len(s) // 2
            split_at = s.rfind(" ", 0, midpoint)     # nearest space before midpoint
            if split_at == -1:
                split_at = s.find(" ", midpoint)      # fallback: nearest space after
            if split_at != -1:
                left = s[:split_at].strip()
                right = s[split_at:].strip()
                if left:
                    sentences.append(left)
                if right:
                    sentences.append(right)
            else:
                sentences.append(s)   # no whitespace at all — keep as-is
        else:
            sentences.append(s)

    return sentences



# Step 1 — detect_structure_boundaries()

```
Change extraction mode from get_text("blocks") to get_text("dict")
  → this gives you font size and font flags per span

For each block on each page:
  → collect the dominant font size of that block
  → collect bold/italic flags

After scanning all pages:
  → compute the body font size (most frequently occurring size)
  → anything with font size > body size by a threshold = heading candidate
  → also flag blocks where ALL spans are bold

For each heading candidate:
  → check it is NOT in the top 7% / bottom 7% zone (skip if it is)
  → check it is NOT a repeat across pages (that's a running header)
  → check it contains at least one real word (not just a number or symbol)
  → if all checks pass → mark this sentence index as a HARD boundary

References/bibliography headings:
  → already handled, keep existing logic, just also register as a hard boundary
  → so the structure detector and the references stopper share the same boundary list
```

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Structure Boundary Detection  [NEW]
# ─────────────────────────────────────────────────────────────────────────────

def detect_structure_boundaries(pdf_path: str, sentences: list[str]) -> set[int]:
    """
    Scans the PDF using get_text("dict") to access font metadata.
    Identifies section headings by font size and bold flag, then maps them
    back to sentence indices so build_coarse_blocks knows where to cut.

    Completely separate from extract_text_from_pdf — does not modify any
    text, does not clean, does not affect the sentence list. Read-only scan.

    A block is a heading candidate if:
      - Its font size exceeds body size by HEADING_SIZE_THRESHOLD, OR
      - All its spans are bold
    Rejected if:
      - Falls in top/bottom 7% zone (mirrors extract_text_from_pdf thresholds)
      - Repeats across 3+ pages (running header)
      - Contains no real alphabetic word
      - Matches a known reference heading (already discarded upstream)
    """
    doc = fitz.open(pdf_path)

    # Pass 1: establish body font size
    all_font_sizes = []

    for page in doc:
        page_height = page.rect.height
        for block in page.get_text("dict")["blocks"]:
            if block.get("type") != 0:
                continue
            if block["bbox"][1] < page_height * 0.07:
                continue
            if block["bbox"][3] > page_height * 0.93:
                continue
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    if span["text"].strip():
                        all_font_sizes.append(round(span["size"], 1))

    if not all_font_sizes:
        doc.close()
        return set()

    body_font_size = Counter(all_font_sizes).most_common(1)[0][0]

    # Pass 2: collect heading candidates, track page repetition
    heading_page_map: dict[str, set[int]] = {}

    for page_num, page in enumerate(doc):
        page_height = page.rect.height

        for block in page.get_text("dict")["blocks"]:
            if block.get("type") != 0:
                continue

            block_y_top    = block["bbox"][1]
            block_y_bottom = block["bbox"][3]

            if block_y_top < page_height * 0.07:
                continue
            if block_y_bottom > page_height * 0.93:
                continue

            span_texts   = []
            is_candidate = False

            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    text = span["text"].strip()
                    if not text:
                        continue
                    span_texts.append(text)
                    is_bold   = bool(span["flags"] & 16)
                    is_larger = round(span["size"], 1) > body_font_size * HEADING_SIZE_THRESHOLD
                    if is_bold or is_larger:
                        is_candidate = True

            if not is_candidate:
                continue

            block_text = " ".join(span_texts).strip()

            if not block_text:
                continue
            if not re.search(r'[a-zA-Z]{2,}', block_text):
                continue

            key = block_text.lower().strip()

            if key in REFERENCE_HEADINGS:
                continue

            heading_page_map.setdefault(key, set()).add(page_num)

    doc.close()

    # Headings on 3+ pages are running headers — discard
    true_headings = {
        text for text, pages in heading_page_map.items()
        if len(pages) < 3
    }

    if not true_headings:
        logging.info(f"'{pdf_path}' — no structural headings detected, full semantic pass will run.")
        return set()

    # Map heading text back to sentence indices
    hard_boundaries: set[int] = set()

    for i, sent in enumerate(sentences):
        sent_lower = sent.lower().strip()
        for heading in true_headings:
            if heading in sent_lower or sent_lower in heading:
                hard_boundaries.add(i)
                break

    logging.info(f"'{pdf_path}' — {len(hard_boundaries)} hard structural boundaries mapped.")
    return hard_boundaries

# Step 2 — build_coarse_blocks()
```
Start with the full sentence list from split_into_sentences()

Walk through sentences:
  → if current sentence index is a HARD boundary from Step 1:
      → close current block, start a new one
  → else:
      → accumulate sentence into current block

After all sentences are walked:
  → you have N coarse blocks, each aligned to a real section boundary

For each coarse block:
  → estimate its token size using the existing len // 4 method
      BUT also count non-ASCII characters separately and weight them higher
      (handles equations and Greek symbols)
  
  → if block is under max_chunk_tokens → mark as FINAL, no further splitting needed
  → if block is over max_chunk_tokens → mark as NEEDS_SEMANTIC_SPLIT
  → if block is under a minimum threshold (e.g. < 3 sentences):
      → merge it into the previous block instead of keeping it alone
```

In [6]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Build Coarse Blocks  [NEW]
# ─────────────────────────────────────────────────────────────────────────────

def _estimate_tokens(sentences: list[str]) -> int:
    """
    Token estimator that weights non-ASCII characters (equations, Greek symbols)
    more heavily than plain ASCII. Private helper used by build_coarse_blocks
    and semantic_refine_large_blocks — not part of the public API.
    """
    total = 0
    for s in sentences:
        ascii_chars     = sum(1 for c in s if ord(c) < 128)
        non_ascii_chars = len(s) - ascii_chars
        total += (ascii_chars // 4) + (non_ascii_chars // 2)
    return total


def build_coarse_blocks(
    sentences: list[str],
    page_refs: list[int],
    hard_boundaries: set[int],
    max_chunk_tokens: int,
) -> list[dict]:
    """
    Splits the sentence list at every hard boundary from detect_structure_boundaries.
    Tags each block needs_split=True if it exceeds max_chunk_tokens — those go
    to semantic_refine_large_blocks. All others are passed straight to
    finalize_chunks_with_overlap.

    Returns internal block dicts — never returned directly to process_pdf_folder
    or add_contextual_retrieval.
    """
    blocks: list[dict] = []
    current_sents: list[str] = []
    current_pages: list[int] = []

    def flush_block() -> None:
        if not current_sents:
            return
        token_est = _estimate_tokens(current_sents)
        blocks.append({
            "sentences":   current_sents[:],
            "page_refs":   current_pages[:],
            "token_est":   token_est,
            "needs_split": token_est > max_chunk_tokens,
        })

    for i, (sent, page) in enumerate(zip(sentences, page_refs)):
        if i in hard_boundaries and current_sents:
            flush_block()
            current_sents = []
            current_pages = []

        current_sents.append(sent)
        current_pages.append(page)

    flush_block()

    # Merge tiny blocks into their predecessor
    merged: list[dict] = []

    for block in blocks:
        if len(block["sentences"]) < MIN_BLOCK_SENTENCES and merged:
            merged[-1]["sentences"].extend(block["sentences"])
            merged[-1]["page_refs"].extend(block["page_refs"])
            merged[-1]["token_est"]   = _estimate_tokens(merged[-1]["sentences"])
            merged[-1]["needs_split"] = merged[-1]["token_est"] > max_chunk_tokens
        else:
            merged.append(block)

    return merged

# Step 3 — semantic_refine_large_blocks()
```
Only called on blocks marked NEEDS_SEMANTIC_SPLIT

For each oversized block:
  → strip any page markers from sentence text before embedding
     (keep a parallel list mapping sentence index → page number)
  → group sentences into sliding windows of size W (e.g. 5 sentences)
     → embed each WINDOW not each individual sentence
     → this reduces embedding calls from N to N/5

  → compute cosine similarity between each consecutive window pair
  → find the local minima in the similarity curve
     → these are candidate semantic boundaries

  → rank candidates by how deep their similarity drop is
  → accept only candidates where the drop exceeds a threshold
     → this avoids cutting on minor topic wobbles

  → from accepted candidates, pick the one closest to the halfway 
     point of the block first, then recurse on each half if still oversized
     → this prevents runaway splitting into too-small pieces

After splitting:
  → re-attach the page references from the parallel page list
  → check resulting sub-blocks against minimum size threshold again
     → merge tiny leftovers into neighbour
```

In [7]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Semantic Refinement of Oversized Blocks  [NEW]
# ─────────────────────────────────────────────────────────────────────────────

def semantic_refine_large_blocks(
    block: dict,
    max_chunk_tokens: int,
    window_size: int = 5,
    similarity_drop_threshold: float = 0.3,
) -> list[dict]:
    """
    Called only on blocks flagged needs_split=True. Embeds sentence windows
    (not individual sentences) in a single batch call, finds topic shift points
    via cosine similarity minima, splits at the midpoint-closest boundary,
    then recurses if either half is still oversized.

    Returns the same internal block dict shape as build_coarse_blocks so
    finalize_chunks_with_overlap can handle both without branching.
    """
    sentences = block["sentences"]
    page_refs = block["page_refs"]

    if len(sentences) < window_size * 2:
        return [block]

    windows = [
        " ".join(sentences[i: i + window_size])
        for i in range(0, len(sentences), window_size)
    ]

    if len(windows) < 2:
        return [block]

    # Single batch encode — no per-sentence API calls
    embeddings = _embedder.encode(windows, convert_to_numpy=True)

    similarities = []
    for i in range(len(embeddings) - 1):
        a = embeddings[i]
        b = embeddings[i + 1]
        cosine = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)
        similarities.append(float(cosine))

    candidates: list[tuple[int, float]] = []

    for i in range(1, len(similarities) - 1):
        is_local_min = (
            similarities[i] < similarities[i - 1]
            and similarities[i] < similarities[i + 1]
        )
        if not is_local_min:
            continue
        drop = min(similarities[i - 1], similarities[i + 1]) - similarities[i]
        if drop > similarity_drop_threshold:
            sent_idx = (i + 1) * window_size
            candidates.append((sent_idx, drop))

    if not candidates:
        logging.debug("semantic_refine: no boundary found above threshold — block kept intact.")
        return [block]

    midpoint = len(sentences) // 2
    candidates.sort(key=lambda x: (abs(x[0] - midpoint), -x[1]))
    split_at = candidates[0][0]

    def make_block(sents: list[str], pages: list[int]) -> dict:
        token_est = _estimate_tokens(sents)
        return {
            "sentences":   sents,
            "page_refs":   pages,
            "token_est":   token_est,
            "needs_split": token_est > max_chunk_tokens,
        }

    left  = make_block(sentences[:split_at], page_refs[:split_at])
    right = make_block(sentences[split_at:], page_refs[split_at:])

    result: list[dict] = []
    for half in (left, right):
        if half["needs_split"]:
            result.extend(
                semantic_refine_large_blocks(half, max_chunk_tokens, window_size, similarity_drop_threshold)
            )
        else:
            result.append(half)

    return result

# Step 4 — finalize_chunks_with_overlap()
```
At this point you have a flat list of refined blocks, all within token bounds

Walk through blocks in order:
  → assign chunk_id sequentially
  → for overlap:
      → take the last overlap_sentences from the PREVIOUS block
      → prepend them to the CURRENT block's sentence list
      → do NOT re-embed these overlap sentences, just carry the text
      → page_start stays as the first non-overlap sentence's page
        (overlap sentences belong to the previous chunk's page range)

Final chunk output shape stays identical to current code:
  → chunk_id, text, page_start, page_end, sentences
  → add one new field: split_method ("structure" or "semantic")
     → useful for debugging and evaluating which path fired
```

In [8]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — Finalize Chunks with Overlap  [NEW]
# ─────────────────────────────────────────────────────────────────────────────

def finalize_chunks_with_overlap(
    blocks: list[dict],
    overlap_sentences: int,
) -> list[dict]:
    """
    Converts internal block dicts into the final public chunk format consumed
    by process_pdf_folder and add_contextual_retrieval.

    Output chunk keys — all keys that add_contextual_retrieval and
    process_pdf_folder depend on are identical to the original code:
      chunk_id               int    — unchanged
      text                   str    — unchanged (what add_contextual_retrieval reads)
      page_start             int    — unchanged (first page of own sentences)
      page_end               int    — unchanged
      sentences              int    — NOW: own sentence count only (excluding overlap)
      sentences_with_overlap int    — NEW: total including overlap sentences
      split_method           str    — NEW: "structure" or "semantic" (debug only)

    The **chunk spread inside add_contextual_retrieval passes all new keys
    through without any collision — "context" and "contextualized_text" are
    the only keys it adds, and neither overlaps with any key here.
    """
    chunks: list[dict] = []

    for i, block in enumerate(blocks):
        own_sents = block["sentences"]
        own_pages = block["page_refs"]

        # page_start belongs to this chunk's own content, not the overlap
        page_start = own_pages[0]

        if i > 0 and overlap_sentences > 0:
            prev          = blocks[i - 1]
            overlap_sents = prev["sentences"][-overlap_sentences:]
            overlap_pages = prev["page_refs"][-overlap_sentences:]
            full_sents    = overlap_sents + own_sents
            full_pages    = overlap_pages + own_pages
        else:
            full_sents = own_sents
            full_pages = own_pages

        chunks.append({
            "chunk_id":               len(chunks),
            "text":                   " ".join(full_sents),
            "page_start":             page_start,
            "page_end":               full_pages[-1],
            "sentences":              len(own_sents),
            "sentences_with_overlap": len(full_sents),
            "split_method":           "semantic" if block.get("was_semantic") else "structure",
        })

    return chunks

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Cosine Similarity
# ─────────────────────────────────────────────────────────────────────────────

def cosine_similarity(a: list[float], b: list[float]) -> float:
    """
    Computes cosine similarity between two embedding vectors.

    Fixes applied:
      - Explicit zero-vector guard: returns 0.0 instead of NaN
        (a near-zero embedding means the sentence had no real content,
         treating it as fully dissimilar forces a chunk boundary — safer)
    """
    a, b = np.array(a), np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom < 1e-10:
        return 0.0  # treat as completely dissimilar → force chunk break
    return float(np.dot(a, b) / denom)

In [9]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — Orchestrator  [UPDATED — internal flow only, return type unchanged]
# ─────────────────────────────────────────────────────────────────────────────

def semantic_chunk_pdf(
    pdf_path: str,
    max_chunk_tokens: int = 512,
    overlap_sentences: int = 2,
) -> list[dict] | None:
    """
    Hybrid chunking pipeline. Returns list[dict] | None — identical return type
    to the original so process_pdf_folder requires zero changes.

    Flow:
      1. extract_text_from_pdf       — unchanged
      2. split_into_sentences        — unchanged, called once, result cached
      3. detect_structure_boundaries — font scan, zero embedding cost
      4. build_coarse_blocks         — split at hard boundaries
      5. semantic_refine_large_blocks — windowed embeddings on oversized blocks only
      6. finalize_chunks_with_overlap — overlap + final public chunk format
    """
    pages = extract_text_from_pdf(pdf_path)
    if pages is None:
        return None

    # Unchanged: join pages with markers for cross-page sentence integrity
    parts = []
    for p in pages:
        parts.append(p["text"])
        parts.append(f"|||PAGE_{p['page_num'] + 1}|||")

    joined_text        = " ".join(parts)
    max_sentence_chars = max_chunk_tokens * 4

    # spaCy called ONCE — result reused by all downstream steps
    raw_sentences = split_into_sentences(joined_text, max_sentence_chars)

    # Unchanged: recover page references from markers
    sentences:    list[str] = []
    page_refs:    list[int] = []
    current_page: int       = pages[0]["page_num"]
    marker_pattern          = re.compile(r'\|\|\|PAGE_(\d+)\|\|\|')

    for sent in raw_sentences:
        marker_match = marker_pattern.search(sent)
        if marker_match:
            page_refs.append(current_page)
            current_page = int(marker_match.group(1))
            clean_sent   = marker_pattern.sub("", sent).strip()
            if clean_sent and re.search(r'[a-zA-Z0-9]', clean_sent):
                sentences.append(clean_sent)
            else:
                page_refs.pop()
        else:
            sentences.append(sent)
            page_refs.append(current_page)

    if not sentences:
        logging.error(f"'{pdf_path}' — text extracted but no sentences found after marker recovery.")
        return None

    hard_boundaries = detect_structure_boundaries(pdf_path, sentences)
    blocks          = build_coarse_blocks(sentences, page_refs, hard_boundaries, max_chunk_tokens)

    needs_semantic = sum(1 for b in blocks if b["needs_split"])
    logging.info(
        f"'{pdf_path}' — {len(blocks)} coarse blocks, "
        f"{needs_semantic} flagged for semantic refinement."
    )

    refined_blocks: list[dict] = []

    for block in blocks:
        if block["needs_split"]:
            sub_blocks = semantic_refine_large_blocks(block, max_chunk_tokens)
            for sb in sub_blocks:
                sb["was_semantic"] = True
            refined_blocks.extend(sub_blocks)
        else:
            block["was_semantic"] = False
            refined_blocks.append(block)

    chunks = finalize_chunks_with_overlap(refined_blocks, overlap_sentences)

    logging.info(f"'{pdf_path}' — pipeline complete. {len(chunks)} final chunks produced.")
    return chunks


In [ ]:

def add_contextual_retrieval(
    chunks: list[dict],
    whole_document: str,
    poll_interval: int = 30,
) -> list[dict]:
    """
    Prepends Claude-generated context to each chunk using the exact prompt
    from the Anthropic Contextual Retrieval blog.

    Updated from sequential API calls to the Anthropic Batch API:
      - All chunk prompts are submitted in a single batch request
      - No rate limit pressure — batch API has no per-minute token limits
      - 50% cheaper than standard API calls
      - Polls every poll_interval seconds until the batch completes
      - Results are mapped back to original chunks via custom_id

    Falls back to empty context (plain text) for any individual chunk
    that errors within the batch — same behaviour as the sequential version.

    Output shape per chunk is identical to before:
      context             — the generated context string alone
      contextualized_text — context prepended to original chunk text
                            (this is what gets embedded and stored)
    """
    MAX_DOC_CHARS = 150_000

    if len(whole_document) > MAX_DOC_CHARS:
        logging.warning(
            f"add_contextual_retrieval — document is {len(whole_document):,} chars, "
            f"truncating to {MAX_DOC_CHARS:,} for Claude context window."
        )
        whole_document = whole_document[:MAX_DOC_CHARS]

    # ── Build one request per chunk ───────────────────────────────────────────
    # custom_id uses chunk_id so results can be mapped back after polling.
    requests = []
    for chunk in chunks:
        prompt = (
            "<document>\n"
            f"{whole_document}\n"
            "</document>\n\n"
            "Here is the chunk we want to situate within the whole document\n"
            "<chunk>\n"
            f"{chunk['text']}\n"
            "</chunk>\n\n"
            "Please give a short succinct context to situate this chunk within "
            "the overall document for the purposes of improving search retrieval "
            "of the chunk. Answer only with the succinct context and nothing else."
        )
        requests.append({
            "custom_id": str(chunk["chunk_id"]),
            "params": {
                "model":      "claude-haiku-4-5-20251001",
                "max_tokens": 100,
                "messages":   [{"role": "user", "content": prompt}],
            },
        })

    # ── Submit batch ──────────────────────────────────────────────────────────
    logging.info(f"add_contextual_retrieval — submitting batch of {len(requests)} requests.")
    batch = anthropic_client.beta.messages.batches.create(requests=requests)
    logging.info(f"add_contextual_retrieval — batch id: {batch.id}, polling every {poll_interval}s.")

    # ── Poll until complete ───────────────────────────────────────────────────
    while True:
        batch = anthropic_client.beta.messages.batches.retrieve(batch.id)
        logging.info(
            f"add_contextual_retrieval — batch status: {batch.processing_status} | "
            f"succeeded: {batch.request_counts.succeeded} | "
            f"errored: {batch.request_counts.errored} | "
            f"in_progress: {batch.request_counts.in_progress}"
        )
        if batch.processing_status == "ended":
            break
        time.sleep(poll_interval)

    # ── Retrieve results and index by custom_id ───────────────────────────────
    results_by_id: dict[str, str] = {}   # chunk_id (str) → context text

    for result in anthropic_client.beta.messages.batches.results(batch.id):
        if result.result.type == "succeeded":
            results_by_id[result.custom_id] = result.result.message.content[0].text.strip()
        else:
            # errored or other non-success — log and fall back to empty context
            logging.warning(
                f"add_contextual_retrieval — chunk {result.custom_id} "
                f"failed in batch (type: {result.result.type}), using raw text."
            )
            results_by_id[result.custom_id] = ""

    # ── Map results back to chunks ────────────────────────────────────────────
    enriched = []
    for chunk in chunks:
        context = results_by_id.get(str(chunk["chunk_id"]), "")
        enriched.append({
            **chunk,
            "context":             context,
            "contextualized_text": f"{context} {chunk['text']}" if context else chunk["text"],
        })

    succeeded = sum(1 for c in enriched if c["context"])
    failed    = len(enriched) - succeeded
    logging.info(
        f"add_contextual_retrieval — complete. "
        f"{succeeded}/{len(enriched)} chunks enriched, {failed} fell back to raw text."
    )

    return enriched

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — Folder Runner
# ─────────────────────────────────────────────────────────────────────────────

def process_pdf_folder(
    folder: str = "./RAG_research_paper",
    add_context: bool = True,
) -> dict[str, list[dict]]:
    """
    Walks a folder of PDFs and returns chunked (+ optionally enriched) results.
    Skipped files are tracked and logged — one bad PDF won't kill the batch.
    """
    pdf_files = list(Path(folder).glob("*.pdf"))

    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found in '{folder}'")

    logging.info(f"Found {len(pdf_files)} PDF(s) in '{folder}'")

    results = {}
    skipped_files = []

    for pdf_file in pdf_files:
        logging.info(f"Processing: {pdf_file.name}")

        try:
            chunks = semantic_chunk_pdf(str(pdf_file))

            if chunks is None:
                skipped_files.append(pdf_file.name)
                logging.warning(f"Skipping '{pdf_file.name}' — no chunks produced.")
                continue

            logging.info(f"'{pdf_file.name}' — {len(chunks)} semantic chunks created.")

            if add_context:
                full_text = " ".join(c["text"] for c in chunks)
                logging.info(f"'{pdf_file.name}' — adding contextual retrieval context via Claude.")
                chunks = add_contextual_retrieval(chunks, full_text)

            results[pdf_file.stem] = chunks

        except Exception as e:
            skipped_files.append(pdf_file.name)
            logging.error(f"Skipping '{pdf_file.name}' — unexpected error: {e}")
            continue

    if skipped_files:
        logging.warning(f"Skipped {len(skipped_files)} file(s): {skipped_files}")

    return results


In [15]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 10 — Vectorstore Initialisation  [UPDATED]
# ─────────────────────────────────────────────────────────────────────────────

def initialise_vectorstore(
    pdf_folder: str      = "./RAG_research_paper",
    persist_path: str    = "./chroma_db",
    collection_name: str = "rag_papers",
    add_context: bool    = True,
    force_rebuild: bool  = False,
) -> "chromadb.Collection":
    """
    Smart vectorstore initialisation — checks what's already on disk and acts
    accordingly. All old helper functions (get_chroma_client,
    get_or_create_collection, embed_chunks, add_chunks_to_collection,
    collection_stats) are replaced with direct ChromaDB calls and our new
    pipeline functions.

    Case 1 — persist_path exists and collection has chunks:
        Returns existing collection immediately. No chunking, no embedding,
        no API calls.

    Case 2 — collection is empty or force_rebuild=True:
        Runs: semantic_chunk_pdf → add_contextual_retrieval → get_embeddings
        → store in ChromaDB.

    Embedding model: get_embeddings() (OpenAI text-embedding-3-small).
    The embedding cache from the old embed_chunks helper is dropped — ChromaDB
    already persists to disk so re-embedding only happens when the collection
    is empty or force_rebuild=True. No separate JSON cache needed.

    Args:
        pdf_folder:      path to folder of PDFs
        persist_path:    where ChromaDB saves its data on disk
        collection_name: name of the ChromaDB collection
        add_context:     whether to run Claude contextual enrichment per chunk
        force_rebuild:   wipe and rebuild even if the collection already exists

    Returns:
        chromadb.Collection — ready to query
    """
    import chromadb

    # ── Create persistent ChromaDB client pointing at local disk path ─────────
    # If persist_path already exists ChromaDB loads it; if not it creates it.
    client = chromadb.PersistentClient(path=persist_path)

    # ── Get or create the collection ──────────────────────────────────────────
    # get_or_create_collection is idempotent — safe to call whether it exists
    # or not. Embeddings are stored as raw vectors; we pass them in manually
    # so ChromaDB does not need its own embedding function.
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"},   # cosine distance for similarity search
    )

    # ── Case 1: collection already populated ──────────────────────────────────
    if collection.count() > 0 and not force_rebuild:
        logging.info(
            f"Vectorstore already exists at '{persist_path}' — "
            f"collection '{collection_name}' has {collection.count()} chunk(s). "
            "Skipping pipeline."
        )
        return collection

    # ── Case 2a: force_rebuild — wipe and recreate ────────────────────────────
    if force_rebuild and collection.count() > 0:
        logging.info(
            f"force_rebuild=True — deleting '{collection_name}' "
            f"({collection.count()} existing chunks) and rebuilding."
        )
        client.delete_collection(name=collection_name)
        collection = client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )

    # ── Case 2b: run the full pipeline ────────────────────────────────────────
    pdf_files = list(Path(pdf_folder).glob("*.pdf"))

    if not pdf_files:
        raise FileNotFoundError(f"No PDFs found in '{pdf_folder}'")

    logging.info(f"Building vectorstore from '{pdf_folder}' — {len(pdf_files)} PDF(s) found.")

    skipped_files = []

    for pdf_file in pdf_files:
        source_name = pdf_file.stem
        logging.info(f"{'─' * 60}")
        logging.info(f"Processing: {pdf_file.name}")

        try:
            # ── Step 1: hybrid semantic chunking ─────────────────────────────
            chunks = semantic_chunk_pdf(str(pdf_file))

            if chunks is None:
                skipped_files.append(pdf_file.name)
                logging.warning(f"Skipping '{pdf_file.name}' — no chunks produced.")
                continue

            logging.info(f"'{pdf_file.name}' — {len(chunks)} chunks produced.")

            # ── Step 2: Claude contextual enrichment (optional) ───────────────
            if add_context:
                full_text = " ".join(c["text"] for c in chunks)
                logging.info(f"'{pdf_file.name}' — running contextual enrichment.")
                chunks = add_contextual_retrieval(chunks, full_text)

            # ── Step 3: embed chunks via OpenAI ───────────────────────────────
            # Use contextualized_text if enrichment ran (richer signal for
            # retrieval), fall back to plain text if add_context=False.
            texts_to_embed = [
                c.get("contextualized_text", c["text"]) for c in chunks
            ]

            logging.info(f"'{pdf_file.name}' — embedding {len(texts_to_embed)} chunks.")
            embeddings = get_embeddings(texts_to_embed)

            # ── Step 4: store in ChromaDB ─────────────────────────────────────
            # IDs must be unique across the entire collection, so prefix with
            # source_name to avoid collisions when multiple PDFs are loaded.
            ids = [f"{source_name}__chunk_{c['chunk_id']}" for c in chunks]

            metadatas = [
                {
                    "source":        source_name,
                    "chunk_id":      c["chunk_id"],
                    "page_start":    c["page_start"],
                    "page_end":      c["page_end"],
                    "sentences":     c["sentences"],
                    "split_method":  c["split_method"],
                    "text":          c["text"],           # original chunk kept for evaluation/reprocessing
                }
                for c in chunks
            ]

            # Store contextualized_text as the document — this is what the LLM
            # receives on retrieval. Falls back to plain text if enrichment was
            # skipped (add_context=False) or failed for a specific chunk.
            documents = [c.get("contextualized_text", c["text"]) for c in chunks]

            collection.add(
                ids=ids,
                embeddings=embeddings,
                metadatas=metadatas,
                documents=documents,
            )

            logging.info(
                f"'{pdf_file.name}' — {len(chunks)} chunks stored in "
                f"collection '{collection_name}'."
            )

        except Exception as e:
            skipped_files.append(pdf_file.name)
            logging.error(f"Skipping '{pdf_file.name}' — unexpected error: {e}")
            continue

    if skipped_files:
        logging.warning(f"Skipped {len(skipped_files)} file(s): {skipped_files}")

    logging.info(
        f"{'═' * 60}\n"
        f"Vectorstore ready — collection '{collection_name}' "
        f"now has {collection.count()} total chunk(s).\n"
        f"{'═' * 60}"
    )

    return collection



In [16]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 11 — Vectorstore Query  [NEW]
# ─────────────────────────────────────────────────────────────────────────────

def query_vectorstore(
    collection: "chromadb.Collection",
    query: str,
    n_results: int = 5,
) -> list[dict]:
    """
    Queries the ChromaDB collection using a plain text question.

    Flow:
      1. Embeds the query using get_embeddings() — same OpenAI model used
         at ingestion time (text-embedding-3-small), ensuring the query
         vector lives in the same space as the stored chunk vectors.
      2. Calls collection.query() with the embedding — ChromaDB returns
         the top n_results chunks by cosine similarity.
      3. Returns a clean ranked list of result dicts.

    Return shape per result:
      rank          — 1-based position (1 = most similar)
      document      — the contextualized_text stored in ChromaDB
                      (context prepended to chunk text — what the LLM should receive)
      text          — original plain chunk text stored in metadata
                      (useful for evaluation: compare against document)
      source        — PDF stem name the chunk came from
      page_start    — start page of the chunk in the original PDF
      page_end      — end page of the chunk in the original PDF
      split_method  — "structure" or "semantic"
      score         — cosine similarity (1 - distance), higher = more relevant

    Standalone and importable — no global state dependencies beyond the
    collection object and get_embeddings().
    """
    logging.info(f"query_vectorstore — embedding query: '{query[:80]}'")

    # Embed query with the same model used at ingestion — single text, one call
    query_embedding = get_embeddings([query])[0]

    raw = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )

    # ChromaDB returns nested lists (one per query) — unwrap the single query
    documents = raw["documents"][0]
    metadatas = raw["metadatas"][0]
    distances = raw["distances"][0]

    results = []
    for rank, (doc, meta, dist) in enumerate(zip(documents, metadatas, distances), start=1):
        results.append({
            "rank":         rank,
            "document":     doc,                          # contextualized_text
            "text":         meta.get("text", ""),         # original plain chunk
            "source":       meta.get("source", ""),
            "page_start":   meta.get("page_start"),
            "page_end":     meta.get("page_end"),
            "split_method": meta.get("split_method", ""),
            "score":        round(1 - dist, 4),           # cosine similarity
        })

    logging.info(
        f"query_vectorstore — returned {len(results)} results "
        f"(top score: {results[0]['score'] if results else 'n/a'})"
    )

    return results



In [17]:

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    """
    Entry point — initialises the vectorstore (loads existing or builds from
    scratch) then runs 4 test queries of increasing complexity against the
    RAG survey paper: "Retrieval-Augmented Generation for Large Language
    Models: A Survey".

    Query progression:
      Q1 — Simple definition   : one concept, expects a direct factual chunk
      Q2 — Mechanism           : how something works, expects a process chunk
      Q3 — Comparison          : two things contrasted, tests retrieval across chunks
      Q4 — Critical/analytical : requires synthesised understanding across the paper
    """
    # Build or load vectorstore
    collection = initialise_vectorstore(
        pdf_folder="./RAG_research_paper",
        persist_path="./chroma_db",
        collection_name="rag_papers",
        add_context=True,
        force_rebuild=False,
    )

    # Four test queries of increasing complexity
    queries = [
        # Q1 — Simple definition: single concept, expects a direct factual answer
        "What is Retrieval-Augmented Generation?",

        # Q2 — Mechanism: how a component works, expects a process-level chunk
        "How does the retriever component select relevant documents in a RAG pipeline?",

        # Q3 — Comparison: two distinct approaches contrasted across the paper
        "What are the differences between sparse retrieval methods like BM25 "
        "and dense retrieval methods like DPR in the context of RAG?",

        # Q4 — Critical/analytical: requires understanding trade-offs discussed
        # across multiple sections — tests whether contextual chunking surfaces
        # the right cross-cutting content
        "What are the key limitations and open research challenges of current "
        "RAG systems when applied to knowledge-intensive tasks, and what "
        "architectural improvements does the survey propose to address them?",
    ]

    for i, query in enumerate(queries, start=1):
        print(f"\n{'=' * 60}")
        print(f"  Q{i}: {query}")
        print(f"{'=' * 60}")

        results = query_vectorstore(collection, query, n_results=3)

        for r in results:
            print(
                f"\n  Rank {r['rank']} | score: {r['score']} | "
                f"source: {r['source']} | "
                f"pages: {r['page_start']}-{r['page_end']} | "
                f"split: {r['split_method']}"
            )
            print(f"  {'-' * 56}")
            # First 300 chars of contextualized document — full text in r['document']
            preview = r["document"][:300].replace("\n", " ")
            print(f"  {preview}...")

        print()


In [18]:
if __name__ == "__main__":
    main()

2026-03-03 21:14:28,104 — INFO — Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-03-03 21:14:28,616 — INFO — Building vectorstore from './RAG_research_paper' — 1 PDF(s) found.
2026-03-03 21:14:28,618 — INFO — ────────────────────────────────────────────────────────────
2026-03-03 21:14:28,618 — INFO — Processing: 2312.10997v5.pdf
2026-03-03 21:14:28,665 — INFO — 'RAG_research_paper\2312.10997v5.pdf' — references section detected at page 17, discarding tail.
c:\Users\Parth\Downloads\Agentic_Rag\venv\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)
2026-03-03 21:14:30,207 — INFO — 'RAG_research_paper\2312.10997v5.pdf' — 8 hard structural boundaries mapped.
2026-03-03

  ✅ Chunk 1/9 enriched


2026-03-03 21:14:36,616 — INFO — HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


  ✅ Chunk 2/9 enriched


2026-03-03 21:14:38,167 — INFO — HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


...


  Rank 1 | score: 0.6516 | source: 2312.10997v5 | pages: 1-1 | split: structure
  --------------------------------------------------------
  This chunk is from the Abstract and Introduction section of a comprehensive survey on Retrieval-Augmented Generation (RAG) for Large Language Models. It establishes the paper's scope by outlining the three main RAG paradigms (Naive, Advanced, and Modular), identifying the core technical components (...

  Rank 2 | score: 0.6502 | source: 2312.10997v5 | pages: 3-3 | split: semantic
  --------------------------------------------------------
  This chunk explains the Naive RAG paradigm, which is the foundational three-step approach to Retrieval-Augmented Generation consisting of indexing, retrieval, and generation. It details the specific limitations and challenges of Naive RAG—including retrieval precision/recall issues, hallucination pr...

  Rank 3 | score: 0.6309 | source: 2312.10997v5 | pages: 1-1 | split: structure
  --------------------------

2026-03-03 21:15:45,342 — INFO — HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-03 21:15:45,361 — INFO — query_vectorstore — returned 3 results (top score: 0.6138)
2026-03-03 21:15:45,366 — INFO — query_vectorstore — embedding query: 'What are the differences between sparse retrieval methods like BM25 and dense re'
2026-03-03 21:15:45,368 — INFO — get_embeddings — batch 1/1 (1 texts)…
2026-03-03 21:15:45,517 — INFO — HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-03 21:15:45,526 — INFO — query_vectorstore — returned 3 results (top score: 0.564)
2026-03-03 21:15:45,527 — INFO — query_vectorstore — embedding query: 'What are the key limitations and open research challenges of current RAG systems'
2026-03-03 21:15:45,529 — INFO — get_embeddings — batch 1/1 (1 texts)…



  Rank 1 | score: 0.6138 | source: 2312.10997v5 | pages: 14-16 | split: semantic
  --------------------------------------------------------
  This chunk is from Section VII (Discussion and Future Prospects) of a comprehensive survey on Retrieval-Augmented Generation (RAG) for Large Language Models. It covers key challenges and future research directions in RAG, including discussion of RAG robustness issues (where irrelevant documents can ...

  Rank 2 | score: 0.5871 | source: 2312.10997v5 | pages: 3-10 | split: semantic
  --------------------------------------------------------
  This chunk from the RAG survey covers the evolution from Naive RAG to Advanced RAG and Modular RAG paradigms, comparing them structurally and functionally. It details key optimization techniques including pre-retrieval processes (query optimization, indexing), post-retrieval processes (re-ranking, c...

  Rank 3 | score: 0.5746 | source: 2312.10997v5 | pages: 14-14 | split: semantic
  -----------------------

2026-03-03 21:15:45,701 — INFO — HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-03 21:15:45,709 — INFO — query_vectorstore — returned 3 results (top score: 0.6884)



  Rank 1 | score: 0.6884 | source: 2312.10997v5 | pages: 14-16 | split: semantic
  --------------------------------------------------------
  This chunk is from Section VII (Discussion and Future Prospects) of a comprehensive survey on Retrieval-Augmented Generation (RAG) for Large Language Models. It covers key challenges and future research directions in RAG, including discussion of RAG robustness issues (where irrelevant documents can ...

  Rank 2 | score: 0.6481 | source: 2312.10997v5 | pages: 1-2 | split: semantic
  --------------------------------------------------------
  This chunk comprises the Introduction and Overview sections of a comprehensive survey on Retrieval-Augmented Generation (RAG) for Large Language Models. It establishes the motivation for RAG by identifying key limitations of LLMs (hallucinations, outdated knowledge, lack of transparency), explains h...

  Rank 3 | score: 0.6237 | source: 2312.10997v5 | pages: 14-14 | split: semantic
  ------------------------

In [ ]:
1. look at the chuks which is generated. 
2. retrieved full chunks and get anwer. 
3. restart the kernal and see if those print and batch is working or not. 


SyntaxError: invalid syntax (2923659731.py, line 1)

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# STEP 10 — Vectorstore Initialisation  [UPDATED]
# ─────────────────────────────────────────────────────────────────────────────

def initialise_vectorstore(
    pdf_folder: str      = "./RAG_research_paper",
    persist_path: str    = "./chroma_db",
    collection_name: str = "rag_papers",
    add_context: bool    = True,
    force_rebuild: bool  = False,
) -> "chromadb.Collection":
    """
    Smart vectorstore initialisation — checks what's already on disk and acts
    accordingly. All old helper functions (get_chroma_client,
    get_or_create_collection, embed_chunks, add_chunks_to_collection,
    collection_stats) are replaced with direct ChromaDB calls and our new
    pipeline functions.

    Case 1 — persist_path exists and collection has chunks:
        Returns existing collection immediately. No chunking, no embedding,
        no API calls.

    Case 2 — collection is empty or force_rebuild=True:
        Runs: semantic_chunk_pdf → add_contextual_retrieval → get_embeddings
        → store in ChromaDB.

    Embedding model: get_embeddings() (OpenAI text-embedding-3-small).
    The embedding cache from the old embed_chunks helper is dropped — ChromaDB
    already persists to disk so re-embedding only happens when the collection
    is empty or force_rebuild=True. No separate JSON cache needed.

    Args:
        pdf_folder:      path to folder of PDFs
        persist_path:    where ChromaDB saves its data on disk
        collection_name: name of the ChromaDB collection
        add_context:     whether to run Claude contextual enrichment per chunk
        force_rebuild:   wipe and rebuild even if the collection already exists

    Returns:
        chromadb.Collection — ready to query
    """
    import chromadb

    # ── Create persistent ChromaDB client pointing at local disk path ─────────
    # If persist_path already exists ChromaDB loads it; if not it creates it.
    client = chromadb.PersistentClient(path=persist_path)

    # ── Get or create the collection ──────────────────────────────────────────
    # get_or_create_collection is idempotent — safe to call whether it exists
    # or not. Embeddings are stored as raw vectors; we pass them in manually
    # so ChromaDB does not need its own embedding function.
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"},   # cosine distance for similarity search
    )

    # ── Case 1: collection already populated ──────────────────────────────────
    if collection.count() > 0 and not force_rebuild:
        logging.info(
            f"Vectorstore already exists at '{persist_path}' — "
            f"collection '{collection_name}' has {collection.count()} chunk(s). "
            "Skipping pipeline."
        )
        return collection

    # ── Case 2a: force_rebuild — wipe and recreate ────────────────────────────
    if force_rebuild and collection.count() > 0:
        logging.info(
            f"force_rebuild=True — deleting '{collection_name}' "
            f"({collection.count()} existing chunks) and rebuilding."
        )
        client.delete_collection(name=collection_name)
        collection = client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )

    # ── Case 2b: run the full pipeline ────────────────────────────────────────
    pdf_files = list(Path(pdf_folder).glob("*.pdf"))

    if not pdf_files:
        raise FileNotFoundError(f"No PDFs found in '{pdf_folder}'")

    logging.info(f"Building vectorstore from '{pdf_folder}' — {len(pdf_files)} PDF(s) found.")

    skipped_files = []

    for pdf_file in pdf_files:
        source_name = pdf_file.stem
        logging.info(f"{'─' * 60}")
        logging.info(f"Processing: {pdf_file.name}")

        try:
            # ── Step 1: hybrid semantic chunking ─────────────────────────────
            chunks = semantic_chunk_pdf(str(pdf_file))

            if chunks is None:
                skipped_files.append(pdf_file.name)
                logging.warning(f"Skipping '{pdf_file.name}' — no chunks produced.")
                continue

            logging.info(f"'{pdf_file.name}' — {len(chunks)} chunks produced.")

            # ── Step 2: Claude contextual enrichment (optional) ───────────────
            if add_context:
                full_text = " ".join(c["text"] for c in chunks)
                logging.info(f"'{pdf_file.name}' — running contextual enrichment.")
                chunks = add_contextual_retrieval(chunks, full_text)

            # ── Step 3: embed chunks via OpenAI ───────────────────────────────
            # Use contextualized_text if enrichment ran (richer signal for
            # retrieval), fall back to plain text if add_context=False.
            texts_to_embed = [
                c.get("contextualized_text", c["text"]) for c in chunks
            ]

            logging.info(f"'{pdf_file.name}' — embedding {len(texts_to_embed)} chunks.")
            embeddings = get_embeddings(texts_to_embed)

            # ── Step 4: store in ChromaDB ─────────────────────────────────────
            # IDs must be unique across the entire collection, so prefix with
            # source_name to avoid collisions when multiple PDFs are loaded.
            ids = [f"{source_name}__chunk_{c['chunk_id']}" for c in chunks]

            metadatas = [
                {
                    "source":        source_name,
                    "chunk_id":      c["chunk_id"],
                    "page_start":    c["page_start"],
                    "page_end":      c["page_end"],
                    "sentences":     c["sentences"],
                    "split_method":  c["split_method"],
                    "text":          c["text"],           # original chunk kept for evaluation/reprocessing
                }
                for c in chunks
            ]

            # Store contextualized_text as the document — this is what the LLM
            # receives on retrieval. Falls back to plain text if enrichment was
            # skipped (add_context=False) or failed for a specific chunk.
            documents = [c.get("contextualized_text", c["text"]) for c in chunks]

            collection.add(
                ids=ids,
                embeddings=embeddings,
                metadatas=metadatas,
                documents=documents,
            )

            logging.info(
                f"'{pdf_file.name}' — {len(chunks)} chunks stored in "
                f"collection '{collection_name}'."
            )

        except Exception as e:
            skipped_files.append(pdf_file.name)
            logging.error(f"Skipping '{pdf_file.name}' — unexpected error: {e}")
            continue

    if skipped_files:
        logging.warning(f"Skipped {len(skipped_files)} file(s): {skipped_files}")

    logging.info(
        f"{'═' * 60}\n"
        f"Vectorstore ready — collection '{collection_name}' "
        f"now has {collection.count()} total chunk(s).\n"
        f"{'═' * 60}"
    )

    return collection

ModuleNotFoundError: No module named 'vectorstore'

In [ ]:
# First run — builds everything from scratch
collection = initialise_vectorstore(pdf_folder="./RAG_research_paper")

# Every run after — detects existing data, skips pipeline entirely
#collection = initialise_vectorstore(pdf_folder="./RAG_research_paper")

# Force a full rebuild (e.g. after changing chunk size or distance metric)
#collection = initialise_vectorstore(pdf_folder="./RAG_research_paper", force_rebuild=True)